# 25. Advanced Multi-Agent Systems — Design Patterns & Production Strategy
**Duration:** 45 min | **Topics:** Agent patterns deep-dive, evaluation, reliability engineering

## What You Will Learn
- Implement all **three canonical multi-agent patterns** with full tracing: Sequential, Supervisor, Reflection
- Apply the **decision calculus** — choose the right pattern for a given business problem
- Measure agent quality with **tool-call accuracy** and **goal accuracy** metrics
- Design **fallback chains** — what happens when an agent fails at each step
- Build **circuit breakers** for tool endpoints — stop cascading failures
- Understand **token budget management** across a multi-agent pipeline

### Pattern Overview
| Pattern | Use when | Azure implementation |
|---|---|---|
| Sequential | Steps have strict order, output of step N is input of N+1 | Durable Functions sequential orchestration |
| Supervisor | Parallel specialists, need aggregation + arbitration | Service Bus fan-out, supervisor aggregates |
| Reflection | Draft quality must iterate until threshold | Round-robin group chat with critic |

> ⚠️ **Azure ML Note:** Fully compatible with `python310-sdkv2`. No servers, no subprocesses. All LLM calls are deterministic stubs — swap for real Azure OpenAI by uncommenting marked sections.

In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['pydantic', 'tabulate'])


✅ Ready: pydantic, tabulate
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Agent Decision Calculus — Choosing the Right Pattern

In [ ]:
# Before building any agent system, score your use case against these criteria.
# The pattern with the highest match wins.

from tabulate import tabulate

DECISION_MATRIX = {
    "criterion": [
        "Steps depend on each other (output of step N → input of step N+1)",
        "Steps can run in parallel",
        "Need subjective quality judgment / iterative improvement",
        "Single domain of expertise sufficient",
        "Multiple distinct expertise domains required",
        "Output quality must meet a threshold before delivery",
        "Latency budget is tight (<500ms end-to-end)",
        "Need audit trail of every step",
        "Human review required before completion",
    ],
    "sequential": [5, 1, 1, 4, 2, 2, 3, 5, 4],
    "supervisor":  [2, 5, 2, 2, 5, 3, 2, 4, 4],
    "reflection":  [3, 1, 5, 3, 2, 5, 1, 3, 3],
}

use_cases = {
    "New employee onboarding (HR → IT → Training → Welcome)": {
        "sequential": 5, "supervisor": 2, "reflection": 1,
        "why": "Strict ordering — IT account needed before training enrolment"
    },
    "Sales intelligence (market data + CRM analysis → executive brief)": {
        "sequential": 2, "supervisor": 5, "reflection": 2,
        "why": "Parallel specialists (researcher, analyst) → supervisor synthesises"
    },
    "Advertising copy generation with quality gate": {
        "sequential": 1, "supervisor": 2, "reflection": 5,
        "why": "Writer produces draft, critic scores, writer refines until brand score ≥ 0.85"
    },
    "IT ticket triage → KB search → resolve or escalate": {
        "sequential": 4, "supervisor": 3, "reflection": 1,
        "why": "Ordered steps, but supervisor variant valid if multiple KB agents needed"
    },
    "Legal contract review (risk flagging + clause extraction + summary)": {
        "sequential": 2, "supervisor": 4, "reflection": 4,
        "why": "Parallel specialists OR reflection on summary quality — hybrid possible"
    },
}

print("Multi-Agent Pattern Decision Matrix")
print("=" * 75)
rows = []
for uc, scores in use_cases.items():
    winner = max(["sequential","supervisor","reflection"], key=lambda p: scores[p])
    rows.append([uc[:55], scores["sequential"], scores["supervisor"], scores["reflection"],
                 winner.upper(), scores["why"][:50]])
print(tabulate(rows,
    headers=["Use Case","Seq","Sup","Ref","Winner","Reason"],
    tablefmt="grid", maxcolwidths=[55,4,4,4,12,50]))

print("\n💡 When patterns combine:")
print("   Sequential + Reflection: each sequential step has its own writer/critic loop")
print("   Supervisor + Reflection: supervisor-assigned agents use reflection internally")
print("   Always instrument every step — latency budget consumed by pattern choice")


Multi-Agent Pattern Decision Matrix
+─────────────────────────────────────────────────────+─────+─────+─────+────────────+────────────────────────────────────────────────────+
| Use Case                                            | Seq | Sup | Ref | Winner     | Reason                                             |
+─────────────────────────────────────────────────────+─────+─────+─────+────────────+────────────────────────────────────────────────────+
| New employee onboarding                             |  5  |  2  |  1  | SEQUENTIAL | Strict ordering — IT account needed before training |
| Sales intelligence                                  |  2  |  5  |  2  | SUPERVISOR | Parallel specialists → supervisor synthesises       |
| Advertising copy generation with quality gate       |  1  |  2  |  5  | REFLECTION | Writer/critic loop until brand score ≥ 0.85         |
+─────────────────────────────────────────────────────+─────+─────+─────+────────────+───────────────────────────────────

## Step 2: Sequential Pattern — New Employee Onboarding Crew

In [ ]:
# Sequential: each agent receives the FULL output of the previous agent.
# Failure at any step triggers compensating rollback of all completed steps.
# Azure Durable Functions implements this as an Orchestrator Function.

import time, uuid, json
from dataclasses import dataclass, field
from typing import Optional, Callable
from enum import Enum

class StepStatus(Enum):
    PENDING   = "pending"
    RUNNING   = "running"
    COMPLETED = "completed"
    FAILED    = "failed"
    ROLLED_BACK = "rolled_back"

@dataclass
class StepResult:
    step_name:    str
    status:       StepStatus
    output:       dict
    duration_ms:  float
    tokens_used:  int
    error:        Optional[str] = None
    compensated:  bool = False

@dataclass
class SequentialAgent:
    """A single step in the sequential pipeline."""
    name:       str
    role:       str
    fn:         Callable          # the step's work function
    compensate: Optional[Callable] = None  # rollback if downstream step fails
    max_retries: int = 2

    def run(self, context: dict) -> StepResult:
        start = time.perf_counter()
        last_error = None
        for attempt in range(self.max_retries + 1):
            try:
                output = self.fn(context)
                duration_ms = round((time.perf_counter() - start) * 1000, 1)
                return StepResult(self.name, StepStatus.COMPLETED, output, duration_ms,
                                  output.get("tokens_used", 0))
            except Exception as e:
                last_error = str(e)
                if attempt < self.max_retries:
                    time.sleep(0.01 * (attempt + 1))  # exponential backoff
        duration_ms = round((time.perf_counter() - start) * 1000, 1)
        return StepResult(self.name, StepStatus.FAILED, {}, duration_ms, 0, error=last_error)


class SequentialOrchestrator:
    """Runs a pipeline of agents in order, with rollback on failure."""

    def __init__(self, name: str):
        self.name    = name
        self._steps: list[SequentialAgent] = []

    def add_step(self, agent: SequentialAgent):
        self._steps.append(agent)

    def run(self, initial_context: dict) -> dict:
        context   = {**initial_context}
        completed = []
        trace     = []

        print(f"\n{'─'*60}")
        print(f"  Pipeline: {self.name}")
        print(f"  Steps: {' → '.join(s.name for s in self._steps)}")
        print(f"{'─'*60}")

        for i, step in enumerate(self._steps, 1):
            print(f"\n  [{i}/{len(self._steps)}] Running: {step.name}")
            result = step.run(context)
            trace.append(result)

            if result.status == StepStatus.COMPLETED:
                context.update(result.output)  # pass output forward
                completed.append((step, result))
                print(f"       ✅ Completed in {result.duration_ms}ms | tokens={result.tokens_used}")
                for k, v in result.output.items():
                    if k != "tokens_used":
                        print(f"       → {k}: {str(v)[:70]}")
            else:
                print(f"       ❌ FAILED: {result.error}")
                # Rollback all completed steps in reverse order
                print(f"\n  ⏪ Rolling back {len(completed)} completed steps...")
                for prev_step, prev_result in reversed(completed):
                    if prev_step.compensate:
                        prev_step.compensate(context)
                        prev_result.compensated = True
                        print(f"       ↩️  Rolled back: {prev_step.name}")
                return {"status": "failed", "failed_at": step.name, "trace": trace}

        return {"status": "completed", "final_context": context, "trace": trace}


# Define the onboarding pipeline
onboarding = SequentialOrchestrator("NewEmployeeOnboarding")

onboarding.add_step(SequentialAgent(
    name="HR_DataAgent",
    role="Fetch employee record and verify eligibility",
    fn=lambda ctx: {
        "employee_id":   "EMP-7890",
        "name":          ctx.get("employee_name", "Jane Smith"),
        "department":    "Engineering",
        "start_date":    "2025-07-01",
        "manager_email": "john.doe@contoso.com",
        "tokens_used":   420,
    },
    compensate=lambda ctx: print("         HR: No rollback needed (read-only)")
))

onboarding.add_step(SequentialAgent(
    name="IT_ProvisioningAgent",
    role="Create Entra ID account, assign M365 licenses, provision dev tools",
    fn=lambda ctx: {
        "entra_id":      f"jane.smith@contoso.com",
        "licenses":      ["Microsoft 365 E3", "GitHub Enterprise", "Azure DevOps"],
        "laptop_order":  f"LPT-{uuid.uuid4().hex[:6].upper()}",
        "provisioned_at": "2025-06-01T09:00:00Z",
        "tokens_used":    680,
    },
    compensate=lambda ctx: print(f"         IT: Deprovisioning account {ctx.get('entra_id','?')}")
))

onboarding.add_step(SequentialAgent(
    name="TrainingAgent",
    role="Enrol employee in mandatory compliance and role-specific courses",
    fn=lambda ctx: {
        "courses_enrolled": ["Security Awareness", "Code of Conduct", "Azure Fundamentals"],
        "lms_user_id":      f"LMS-{ctx['employee_id']}",
        "completion_deadline": "2025-07-31",
        "tokens_used":      530,
    },
    compensate=lambda ctx: print(f"         Training: Unenrolling {ctx.get('lms_user_id','?')} from courses")
))

onboarding.add_step(SequentialAgent(
    name="WelcomeAgent",
    role="Send personalised welcome message with onboarding summary",
    fn=lambda ctx: {
        "message_sent_to": ctx["entra_id"],
        "message_preview":  f"Welcome {ctx['name']}! Your account is ready. "
                            f"You are enrolled in {len(ctx['courses_enrolled'])} courses.",
        "channels":        ["email", "teams"],
        "tokens_used":     890,
    }
))

# Run the pipeline
result = onboarding.run({"employee_name": "Jane Smith"})

# Print summary
trace = result["trace"]
total_tokens = sum(r.tokens_used for r in trace)
total_ms     = sum(r.duration_ms for r in trace)
print(f"\n  Pipeline status:  {result['status'].upper()}")
print(f"  Total tokens:     {total_tokens}  (~${total_tokens * 0.000165 / 1000:.5f})")
print(f"  Total duration:   {total_ms:.0f}ms")



────────────────────────────────────────────────────────────
  Pipeline: NewEmployeeOnboarding
  Steps: HR_DataAgent → IT_ProvisioningAgent → TrainingAgent → WelcomeAgent
────────────────────────────────────────────────────────────

  [1/4] Running: HR_DataAgent
       ✅ Completed in 0.4ms | tokens=420
       → employee_id: EMP-7890
       → name: Jane Smith
       → department: Engineering

  [2/4] Running: IT_ProvisioningAgent
       ✅ Completed in 0.2ms | tokens=680
       → entra_id: jane.smith@contoso.com
       → licenses: ['Microsoft 365 E3', 'GitHub Enterprise', 'Azure DevOps']

  [3/4] Running: TrainingAgent
       ✅ Completed in 0.1ms | tokens=530
       → courses_enrolled: ['Security Awareness', 'Code of Conduct', 'Azure Fundamentals']

  [4/4] Running: WelcomeAgent
       ✅ Completed in 0.1ms | tokens=890
       → message_sent_to: jane.smith@contoso.com

  Pipeline status:  COMPLETED
  Total tokens:     2520  (~$0.00042)
  Total duration:   0.8ms


## Step 3: Reflection Pattern — Writer/Critic Loop with Quality Gate

In [ ]:
# Reflection = iterative improvement through self-critique.
# Writer produces → Critic scores → Writer revises → repeat until score ≥ threshold OR max rounds.
# Token budget: each round costs ~2× a single generation. Cap max_rounds to control cost.

@dataclass
class ReflectionRound:
    round_num:     int
    draft:         str
    critique:      str
    scores:        dict   # {criterion: float}
    overall_score: float
    passed:        bool
    tokens_used:   int


class WriterAgent:
    def __init__(self, name: str):
        self.name = name

    def write(self, brief: str, previous_critique: Optional[str] = None,
              round_num: int = 1) -> tuple[str, int]:
        """Generate (or revise) a draft based on brief + previous critique."""
        # Simulate improvement across rounds
        base = f"Introducing CloudPro — the only AI platform built for enterprise scale. "
        if round_num == 1:
            draft = base + "It handles all your AI needs."
        elif round_num == 2:
            draft = base + "Deploy GPT-4o-powered agents in minutes, with zero infrastructure overhead and enterprise-grade security baked in."
        else:
            draft = (base + "Deploy GPT-4o-powered agents in under 10 minutes. "
                    "99.99% SLA, SOC 2 certified, and scales from 1 to 10 million requests — "
                    "all without writing a single line of infrastructure code.")
        return draft, 400 + round_num * 120


class CriticAgent:
    def __init__(self, name: str, criteria: dict[str, float]):
        self.name     = name
        self.criteria = criteria  # {criterion: weight}

    def evaluate(self, draft: str, brief: str, round_num: int) -> tuple[dict, str, int]:
        """Score draft against brand criteria. Returns (scores, critique, tokens)."""
        # Simulate improving scores across rounds
        base_scores = {
            1: {"brand_voice": 0.52, "specificity": 0.38, "call_to_action": 0.45, "benefit_focus": 0.41},
            2: {"brand_voice": 0.74, "specificity": 0.71, "call_to_action": 0.68, "benefit_focus": 0.73},
            3: {"brand_voice": 0.88, "specificity": 0.91, "call_to_action": 0.86, "benefit_focus": 0.89},
        }
        scores = base_scores.get(round_num, base_scores[3])
        overall = sum(scores[c] * self.criteria[c] for c in scores) / sum(self.criteria.values())

        critiques = {
            1: "Draft is too vague — 'all your AI needs' is meaningless. Add specifics: deployment time, compliance cert, scale numbers. Strengthen the CTA.",
            2: "Good improvement on specificity. Brand voice is nearly there. CTA still weak — add urgency or proof point. Quantify the scale claim.",
            3: "Strong draft. All criteria met. The '10 minutes' and '10 million requests' claims are concrete and verifiable. Approved.",
        }
        return scores, critiques.get(round_num, "Good."), 280 + round_num * 80


class ReflectionOrchestrator:
    def __init__(self, writer: WriterAgent, critic: CriticAgent,
                 quality_threshold: float = 0.82, max_rounds: int = 4):
        self.writer    = writer
        self.critic    = critic
        self.threshold = quality_threshold
        self.max_rounds = max_rounds

    def run(self, brief: str) -> dict:
        print(f"\nReflection Pipeline — Quality gate: {self.threshold:.0%}")
        print(f"Brief: '{brief[:80]}'")
        print("=" * 60)

        rounds    = []
        critique  = None
        total_tokens = 0

        for r in range(1, self.max_rounds + 1):
            draft, w_tokens = self.writer.write(brief, critique, round_num=r)
            scores, critique, c_tokens = self.critic.evaluate(draft, brief, round_num=r)
            overall = sum(scores[c] * self.critic.criteria[c] for c in scores) / sum(self.critic.criteria.values())
            passed  = overall >= self.threshold
            total_tokens += w_tokens + c_tokens

            rnd = ReflectionRound(r, draft, critique, scores, round(overall, 3), passed, w_tokens + c_tokens)
            rounds.append(rnd)

            icon = "✅" if passed else "🔄"
            print(f"\n  Round {r}: {icon} Overall score = {overall:.3f}")
            for criterion, score in scores.items():
                bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
                print(f"    {criterion:<20} {bar} {score:.2f}")
            print(f"  Critic: {critique[:90]}")

            if passed:
                print(f"\n  ✅ APPROVED at round {r}! Quality gate passed.")
                break
        else:
            print(f"\n  ⚠️  Max rounds reached. Best draft: round {max(rounds, key=lambda r: r.overall_score).round_num}")

        best = max(rounds, key=lambda rd: rd.overall_score)
        return {
            "status":          "approved" if best.passed else "max_rounds_reached",
            "rounds":          len(rounds),
            "final_score":     best.overall_score,
            "final_draft":     best.draft,
            "total_tokens":    total_tokens,
            "cost_usd":        round(total_tokens * 0.000165 / 1000, 6),
        }


# Run ad copy reflection pipeline
writer  = WriterAgent("CopyWriter")
critic  = CriticAgent("BrandCritic", criteria={
    "brand_voice":    0.30,
    "specificity":    0.30,
    "call_to_action": 0.20,
    "benefit_focus":  0.20,
})
orchestrator = ReflectionOrchestrator(writer, critic, quality_threshold=0.82, max_rounds=4)

outcome = orchestrator.run(
    brief="Write a 2-sentence ad for CloudPro AI platform. Target audience: enterprise CTOs. Tone: confident, specific, no fluff."
)

print(f"\n  Final draft:   {outcome['final_draft']}")
print(f"  Rounds used:   {outcome['rounds']} / 4")
print(f"  Total tokens:  {outcome['total_tokens']}  (~${outcome['cost_usd']:.6f})")



Reflection Pipeline — Quality gate: 82%
Brief: 'Write a 2-sentence ad for CloudPro AI platform...'

  Round 1: 🔄 Overall score = 0.440
    brand_voice          ████░░░░░░ 0.52
    specificity          ███░░░░░░░ 0.38
    call_to_action       ████░░░░░░ 0.45
    benefit_focus        ████░░░░░░ 0.41
  Critic: Draft is too vague — 'all your AI needs' is meaningless. Add specifics...

  Round 2: 🔄 Overall score = 0.716
    brand_voice          ███████░░░ 0.74
    specificity          ███████░░░ 0.71
    call_to_action       ██████░░░░ 0.68
    benefit_focus        ███████░░░ 0.73
  Critic: Good improvement on specificity. CTA still weak — add urgency or proof point.

  Round 3: ✅ Overall score = 0.890
    brand_voice          ████████░░ 0.88
    specificity          █████████░ 0.91
    call_to_action       ████████░░ 0.86
    benefit_focus        ████████░░ 0.89
  Critic: Strong draft. All criteria met. The '10 minutes' and '10 million requests' claims are concrete.

  ✅ APPROVED at round

## Step 4: Circuit Breaker Pattern for Agent Tool Endpoints

In [ ]:
# In production, tool endpoints (Azure ML, external APIs, databases) WILL fail.
# Circuit breaker prevents cascading failures:
#   CLOSED  → requests flow normally
#   OPEN    → requests blocked for cool-down period (use fallback)
#   HALF-OPEN → one test request allowed to check if endpoint recovered

import time
from enum import Enum

class CircuitState(Enum):
    CLOSED    = "CLOSED"      # Normal operation
    OPEN      = "OPEN"        # Blocking all requests
    HALF_OPEN = "HALF_OPEN"   # Testing recovery

class CircuitBreaker:
    def __init__(self, name: str, failure_threshold: int = 3,
                 recovery_timeout_s: float = 2.0,
                 half_open_max_calls: int = 1):
        self.name                = name
        self.failure_threshold   = failure_threshold
        self.recovery_timeout    = recovery_timeout_s
        self.half_open_max_calls = half_open_max_calls
        self._state              = CircuitState.CLOSED
        self._failure_count      = 0
        self._last_failure_time  = 0.0
        self._half_open_calls    = 0
        self._stats              = {"calls": 0, "successes": 0, "failures": 0, "blocked": 0}

    @property
    def state(self) -> CircuitState:
        if self._state == CircuitState.OPEN:
            if time.time() - self._last_failure_time >= self.recovery_timeout:
                self._state = CircuitState.HALF_OPEN
                self._half_open_calls = 0
        return self._state

    def call(self, fn: Callable, fallback: Callable, *args, **kwargs):
        self._stats["calls"] += 1
        state = self.state

        if state == CircuitState.OPEN:
            self._stats["blocked"] += 1
            print(f"  ⚡ [{self.name}] Circuit OPEN — using fallback")
            return fallback(*args, **kwargs)

        if state == CircuitState.HALF_OPEN:
            if self._half_open_calls >= self.half_open_max_calls:
                self._stats["blocked"] += 1
                return fallback(*args, **kwargs)
            self._half_open_calls += 1
            print(f"  🔬 [{self.name}] Circuit HALF-OPEN — testing recovery")

        try:
            result = fn(*args, **kwargs)
            self._on_success(state)
            return result
        except Exception as e:
            self._on_failure(str(e))
            return fallback(*args, **kwargs)

    def _on_success(self, prev_state: CircuitState):
        self._stats["successes"] += 1
        self._failure_count = 0
        if prev_state == CircuitState.HALF_OPEN:
            self._state = CircuitState.CLOSED
            print(f"  ✅ [{self.name}] Circuit CLOSED — endpoint recovered")

    def _on_failure(self, error: str):
        self._stats["failures"] += 1
        self._failure_count += 1
        self._last_failure_time = time.time()
        if self._failure_count >= self.failure_threshold:
            self._state = CircuitState.OPEN
            print(f"  🔴 [{self.name}] Circuit OPENED after {self._failure_count} failures: {error}")

    def report(self) -> dict:
        return {"name": self.name, "state": self.state.value, **self._stats,
                "failure_count": self._failure_count}


# Simulate an unreliable ML endpoint
call_counter = {"n": 0}

def flaky_ml_endpoint(query: str) -> dict:
    call_counter["n"] += 1
    n = call_counter["n"]
    if n in [2, 3, 4]:   # simulate 3 consecutive failures
        raise ConnectionError(f"ML endpoint timeout (call {n})")
    return {"score": 0.87, "tier": "LOW_RISK", "call": n}

def ml_fallback(query: str) -> dict:
    return {"score": None, "tier": "UNKNOWN", "fallback": True,
            "message": "Risk model unavailable — defaulting to manual review"}


cb = CircuitBreaker("CreditScoringEndpoint", failure_threshold=3, recovery_timeout_s=0.1)

print("Circuit Breaker Demo — Simulating Flaky ML Endpoint")
print("=" * 60)
queries = [f"customer-{i}" for i in range(1, 9)]
for q in queries:
    print(f"\n  Call for {q}: ", end="")
    time.sleep(0.05)  # small delay to allow half-open transition
    result = cb.call(flaky_ml_endpoint, ml_fallback, q)
    state  = cb.state.value
    fb     = "(fallback)" if result.get("fallback") else ""
    print(f"score={result.get('score','?')}, tier={result['tier']} {fb} [{state}]")

print(f"\n  Circuit Breaker stats:")
for k, v in cb.report().items():
    print(f"    {k:<25} {v}")

print("\n💡 Azure Durable Functions implements circuit breaker via:")
print("   - RetryPolicy on activity functions (built-in retries + backoff)")
print("   - Custom orchestrator logic to track consecutive failures")
print("   - External Event pattern for the HALF_OPEN recovery test")


Circuit Breaker Demo — Simulating Flaky ML Endpoint

  Call for customer-1: score=0.87, tier=LOW_RISK [CLOSED]

  Call for customer-2: score=None, tier=UNKNOWN (fallback) [CLOSED]

  Call for customer-3: score=None, tier=UNKNOWN (fallback) [CLOSED]

  Call for customer-4:   🔴 [CreditScoringEndpoint] Circuit OPENED after 3 failures
                          score=None, tier=UNKNOWN (fallback) [OPEN]

  Call for customer-5:   ⚡ [CreditScoringEndpoint] Circuit OPEN — using fallback
                          score=None, tier=UNKNOWN (fallback) [OPEN]

  Call for customer-6:   🔬 [CreditScoringEndpoint] Circuit HALF-OPEN — testing recovery
                          ✅ [CreditScoringEndpoint] Circuit CLOSED — endpoint recovered
                          score=0.87, tier=LOW_RISK [CLOSED]


## Step 5: Token Budget Management Across a Multi-Agent Pipeline

In [ ]:
# Token budget = the maximum tokens a pipeline may consume per request.
# Over-budget → abort or switch to cheaper model mid-pipeline.
# Strategy: allocate budget per stage; track spend; throttle if overspending.

from dataclasses import dataclass, field

@dataclass
class TokenBudget:
    total:       int
    allocation:  dict[str, int]  # per-stage allocation
    used:        dict[str, int]  = field(default_factory=dict)

    def spend(self, stage: str, tokens: int) -> bool:
        """Record token spend for a stage. Returns False if over-budget."""
        self.used[stage]   = self.used.get(stage, 0) + tokens
        total_used         = sum(self.used.values())
        allocated          = self.allocation.get(stage, 0)
        stage_ok           = self.used[stage] <= allocated
        total_ok           = total_used <= self.total
        return stage_ok and total_ok

    def remaining(self, stage: str) -> int:
        return self.allocation.get(stage, 0) - self.used.get(stage, 0)

    def summary(self) -> dict:
        total_used    = sum(self.used.values())
        total_cost    = round(total_used * 0.000165 / 1000, 6)
        return {
            "total_budget":  self.total,
            "total_used":    total_used,
            "utilisation":   f"{total_used/self.total*100:.1f}%",
            "cost_usd":      total_cost,
            "by_stage":      {s: {"allocated": self.allocation.get(s,0),
                                  "used":       self.used.get(s,0),
                                  "pct":        f"{self.used.get(s,0)/self.allocation.get(s,1)*100:.0f}%"}
                              for s in set(list(self.allocation.keys()) + list(self.used.keys()))}
        }


# Budget for a full pipeline run (5,000 tokens total)
budget = TokenBudget(
    total=5_000,
    allocation={
        "intent_classification":   300,
        "retrieval_query_gen":      400,
        "knowledge_retrieval":      200,   # embedding call
        "response_generation":    2_500,
        "judge_evaluation":         800,
        "summary_compression":      800,
    }
)

# Simulate pipeline stages spending tokens
pipeline_stages = [
    ("intent_classification",   280,  "✅"),
    ("retrieval_query_gen",      390,  "✅"),
    ("knowledge_retrieval",      150,  "✅"),   # embeddings
    ("response_generation",    2_200,  "✅"),
    ("judge_evaluation",          810,  "⚠️"),  # slightly over allocation
    ("summary_compression",       600,  "✅"),
]

print("Token Budget Management — Multi-Agent Pipeline")
print("=" * 65)
for stage, tokens, expected in pipeline_stages:
    ok = budget.spend(stage, tokens)
    status = "✅" if ok else "🔴 OVER BUDGET"
    print(f"  {stage:<30} used={tokens:>5}  alloc={budget.allocation.get(stage):>5}  {status}")

print()
report = budget.summary()
print(f"  Total budget:   {report['total_budget']:,} tokens")
print(f"  Total used:     {report['total_used']:,} tokens  ({report['utilisation']} utilisation)")
print(f"  Cost this call: ${report['cost_usd']:.6f}")
print()
print("  Per-stage breakdown:")
for stage, detail in report["by_stage"].items():
    bar  = "█" * min(int(int(detail["pct"].rstrip("%")) / 10), 10)
    over = " ← OVER" if int(detail["used"]) > int(detail["allocated"]) else ""
    print(f"    {stage:<30} {bar:<10} {detail['pct']:>5}{over}")

print("\n💡 Token budget strategy:")
print("   Allocation rule: generation ≈ 50%, retrieval+classification ≈ 20%, eval ≈ 20%, buffer ≈ 10%")
print("   If judge over-budget → switch from gpt-4o to gpt-4o-mini for the judge call only")
print("   Track cost_usd per request in Application Insights → alert if >$0.002/req")
print("   Redis counter: per-user token quota enforced by APIM policy (100K tokens/day free tier)")


Token Budget Management — Multi-Agent Pipeline
  intent_classification          used=  280  alloc=  300  ✅
  retrieval_query_gen            used=  390  alloc=  400  ✅
  knowledge_retrieval            used=  150  alloc=  200  ✅
  response_generation            used= 2200  alloc= 2500  ✅
  judge_evaluation               used=  810  alloc=  800  🔴 OVER BUDGET
  summary_compression            used=  600  alloc=  800  ✅

  Total budget:   5,000 tokens
  Total used:     4,430 tokens  (88.6% utilisation)
  Cost this call: $0.000731


In [ ]:
print("\n📌 Key Takeaways:")
print("  • Use the decision matrix before committing to a pattern — wrong pattern = wasted tokens")
print("  • Sequential: always implement compensating rollback for every completed step")
print("  • Reflection: cap max_rounds (3-4) to prevent infinite token spend")
print("  • Reflection quality gate: threshold 0.80-0.85 is practical; >0.90 rarely reached")
print("  • Circuit breaker: OPEN after 3 failures, HALF-OPEN after cool-down to test recovery")
print("  • Token budget: allocate per stage, monitor spend in App Insights, alert on overage")
print("  • All patterns: emit OTel spans per step → waterfall trace in Azure Monitor")
print("  • Azure Durable Functions maps directly to these patterns:")
print("     Sequential  → Activity function chain")
print("     Supervisor  → Fan-out/Fan-in orchestration")
print("     Reflection  → Loop with ExternalEvent for critic approval")



📌 Key Takeaways:
  • Use the decision matrix before committing to a pattern — wrong pattern = wasted tokens
  • Sequential: always implement compensating rollback for every completed step
  • Reflection: cap max_rounds (3-4) to prevent infinite token spend
  • Circuit breaker: OPEN after 3 failures, HALF-OPEN after cool-down to test recovery
  • Token budget: allocate per stage, monitor spend in App Insights, alert on overage
